# 1. Web Scraping

Download three tables for every NBA season from 1991 to 2025 from [basketball-reference.com](https://www.basketball-reference.com):

| Data | Page | Used as |
|---|---|---|
| MVP voting | `awards/awards_{year}.html` | Target (vote share) |
| Player per-game stats | `leagues/NBA_{year}_per_game.html` | Features |
| Team standings | `leagues/NBA_{year}_standings.html` | Features (team success) |

**Scraping rules**
- The site allows **max 20 requests per minute** (more = blocked for 1 hour), so we wait 7 seconds between requests.
- Every page is saved as raw HTML first, then parsed. If parsing has a bug, we fix it without downloading again.
- Pages that already exist on disk are skipped, so this notebook can be re-run safely.

## Setup

In [1]:
import os
import time
import requests
import pandas as pd
from io import StringIO
from bs4 import BeautifulSoup

YEARS = range(1991, 2026)
for folder in ["mvp", "player", "team"]:
    os.makedirs(folder, exist_ok=True)

In [2]:
def download(url, path):
    """Save a page as raw HTML. Skips files that already exist."""
    if os.path.exists(path):
        return
    page = requests.get(url)
    if page.status_code == 429:
        raise RuntimeError("Rate limited (HTTP 429). Wait 1 hour before trying again.")
    page.raise_for_status()
    with open(path, "wb") as f:    # raw bytes keep the UTF-8 encoding (e.g. "Jokić")
        f.write(page.content)
    print("Saved", path)
    time.sleep(7)                  # stay under 20 requests/minute


def read_table(path, table_id, remove_row_class=None):
    """Parse one HTML table into a DataFrame."""
    with open(path, encoding="utf-8") as f:
        soup = BeautifulSoup(f.read(), "html.parser")
    if remove_row_class:
        extra_header = soup.find("tr", class_=remove_row_class)
        if extra_header:
            extra_header.decompose()
    table = soup.find(id=table_id)
    return pd.read_html(StringIO(str(table)))[0]

## MVP voting

The MVP table has an extra grouped header row (`over_header`) that we remove before parsing.

In [3]:
for year in YEARS:
    download(f"https://www.basketball-reference.com/awards/awards_{year}.html", f"mvp/{year}.html")

dfs = []
for year in YEARS:
    mvp = read_table(f"mvp/{year}.html", "mvp", remove_row_class="over_header")
    mvp["Year"] = year
    dfs.append(mvp)

mvps = pd.concat(dfs, ignore_index=True)
mvps.to_csv("mvps.csv", index=False)
print(mvps.shape)
mvps.head()

(520, 21)


,Rank,Player,Age,Tm,First,Pts Won,Pts Max,Share,G,MP,...,TRB,AST,STL,BLK,FG%,3P%,FT%,WS,WS/48,Year
0,1,Michael Jordan,27,CHI,77,891,960,0.928,82,37.0,...,6.0,5.5,2.7,1.0,0.539,0.312,0.851,20.3,0.321,1991
1,2,Magic Johnson,31,LAL,10,497,960,0.518,79,37.1,...,7.0,12.5,1.3,0.2,0.477,0.320,0.906,15.4,0.251,1991
2,3,David Robinson,25,SAS,6,476,960,0.496,82,37.7,...,13.0,2.5,1.5,3.9,0.552,0.143,0.762,17.0,0.264,1991
3,4,Charles Barkley,27,PHI,2,222,960,0.231,67,37.3,...,10.1,4.2,1.6,0.5,0.570,0.284,0.722,13.4,0.258,1991
4,5,Karl Malone,27,UTA,0,142,960,0.148,82,40.3,...,11.8,3.3,1.1,1.0,0.527,0.286,0.770,15.5,0.225,1991


## Player per-game stats

The original tutorial needed Selenium because this table was loaded with JavaScript. It is now part of the static HTML, so `requests` is enough.

In [4]:
for year in YEARS:
    download(f"https://www.basketball-reference.com/leagues/NBA_{year}_per_game.html", f"player/{year}.html")

dfs = []
for year in YEARS:
    player = read_table(f"player/{year}.html", "per_game_stats", remove_row_class="thead")
    player["Year"] = year
    dfs.append(player)

players = pd.concat(dfs, ignore_index=True)
players.to_csv("players.csv", index=False)
print(players.shape)
players.head()

(21340, 32)


,Rk,Player,Age,Team,Pos,G,GS,MP,FG,FGA,...,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,Awards,Year
0,1,Michael Jordan,27,CHI,SG,82,82,37.0,12.1,22.4,...,4.6,6.0,5.5,2.7,1.0,2.5,2.8,31.5,"MVP-1,DPOY-7,AS,NBA1,DEF1",1991
1,2,Karl Malone,27,UTA,PF,82,82,40.3,10.3,19.6,...,8.9,11.8,3.3,1.1,1.0,3.0,3.3,29.0,"MVP-5,AS,NBA1",1991
2,3,Bernard King,34,WSB,SF,64,64,37.5,11.1,23.6,...,3.2,5.0,4.6,0.9,0.3,4.0,2.9,28.4,"MVP-16,AS,NBA3",1991
3,4,Charles Barkley,27,PHI,SF,67,67,37.3,9.9,17.4,...,6.3,10.1,4.2,1.6,0.5,3.1,2.6,27.6,"MVP-4,AS,NBA1",1991
4,5,Patrick Ewing,28,NYK,C,81,81,38.3,10.4,20.3,...,8.8,11.2,3.0,1.0,3.2,3.6,3.5,26.6,"MVP-11,DPOY-7,AS,NBA2",1991


## Team standings

Each season has two tables, one per conference. We stack them and rename the conference column to `Team`.

In [5]:
for year in YEARS:
    download(f"https://www.basketball-reference.com/leagues/NBA_{year}_standings.html", f"team/{year}.html")

dfs = []
for year in YEARS:
    for table_id, conference in [("divs_standings_E", "Eastern Conference"),
                                 ("divs_standings_W", "Western Conference")]:
        team = read_table(f"team/{year}.html", table_id, remove_row_class="thead")
        team = team.rename(columns={conference: "Team"})
        team["Year"] = year
        dfs.append(team)

teams = pd.concat(dfs, ignore_index=True)
teams.to_csv("teams.csv", index=False)
print(teams.shape)
teams.head()

(1173, 9)


,Team,W,L,W/L%,GB,PS/G,PA/G,SRS,Year
0,Boston Celtics*,56,26,.683,—,111.5,105.7,5.22,1991
1,Philadelphia 76ers*,44,38,.537,12.0,105.4,105.6,-0.39,1991
2,New York Knicks*,39,43,.476,17.0,103.1,103.3,-0.43,1991
3,Washington Bullets,30,52,.366,26.0,101.4,106.4,-4.84,1991
4,New Jersey Nets,26,56,.317,30.0,102.9,107.5,-4.53,1991


## Output

| File | Rows | Description |
|---|---|---|
| `mvps.csv` | one per MVP vote-getter | MVP voting results |
| `players.csv` | one per player per team | Per-game stats (traded players have several rows) |
| `teams.csv` | one per team + division header rows | Standings |

Cleaning happens in `02_data_cleaning.ipynb`.